# AMD Quarterly Stock Price Prediction Model

Predict AMD's **next-quarter price return** using SEC XBRL fundamental data + price momentum.

| | |
|---|---|
| **Data** | SEC EDGAR XBRL API (2012–present) + yfinance price history |
| **Granularity** | Fiscal quarter (~13 weeks) |
| **Models** | Ridge · Random Forest · XGBoost |
| **Validation** | Walk-forward (no look-ahead) — train on past, test on next quarter |
| **Target** | Next quarter's price return (%) |

Run cells top-to-bottom. Cell 2–3 make network requests; the rest is local computation.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import requests
import yfinance as yf

from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")

# ── Config ────────────────────────────────────────────────────────────────────
TICKER       = "AMD"
CIK          = "CIK0000002488"
USER_AGENT   = "sunjialin sunjialin@terraquant.cn"
RANDOM_STATE = 42
MIN_TRAIN    = 20   # minimum quarters before first prediction

print("Libraries loaded. Config:")
print(f"  Ticker : {TICKER}")
print(f"  CIK    : {CIK}")
print(f"  Min training window: {MIN_TRAIN} quarters")

In [ ]:
# ── SEC XBRL helper functions ─────────────────────────────────────────────────

def fetch_gaap(cik, user_agent):
    """Download all GAAP XBRL facts from SEC EDGAR company-facts API."""
    url = f"https://data.sec.gov/api/xbrl/companyfacts/{cik}.json"
    r = requests.get(url, headers={"User-Agent": user_agent}, timeout=60)
    r.raise_for_status()
    return r.json()["facts"]["us-gaap"]


def flow_qtr(gaap, concept, col):
    """
    Income-statement concept → standalone single-quarter series.
    Filters to entries where period duration is 70–105 days (one quarter).
    Handles both 10-Q (Q1–Q3) and 10-K (Q4) filings.
    """
    if concept not in gaap:
        return pd.Series(dtype=float, name=col)
    uk = "USD" if "USD" in gaap[concept]["units"] else next(iter(gaap[concept]["units"]))
    df = pd.DataFrame(gaap[concept]["units"][uk])
    if "start" not in df.columns:
        return pd.Series(dtype=float, name=col)
    df["start"] = pd.to_datetime(df["start"])
    df["end"]   = pd.to_datetime(df["end"])
    df["filed"] = pd.to_datetime(df["filed"])
    df["dur"]   = (df["end"] - df["start"]).dt.days
    q = df[df["form"].isin(["10-Q", "10-K"]) & df["dur"].between(70, 105)].copy()
    q = q.sort_values("filed").drop_duplicates("end", keep="last")
    return q.set_index("end")["val"].rename(col).sort_index().astype(float)


def stock_qtr(gaap, concept, col):
    """
    Balance-sheet concept → point-in-time quarterly series.
    No duration filter (instantaneous values have no start date).
    """
    if concept not in gaap:
        return pd.Series(dtype=float, name=col)
    uk = "USD" if "USD" in gaap[concept]["units"] else next(iter(gaap[concept]["units"]))
    df = pd.DataFrame(gaap[concept]["units"][uk])
    df["end"]   = pd.to_datetime(df["end"])
    df["filed"] = pd.to_datetime(df["filed"])
    q = df[df["form"].isin(["10-Q", "10-K"])].copy()
    q = q.sort_values("filed").drop_duplicates("end", keep="last")
    return q.set_index("end")["val"].rename(col).sort_index().astype(float)


# ── Download ──────────────────────────────────────────────────────────────────
print(f"Downloading {TICKER} XBRL data from SEC EDGAR...")
gaap = fetch_gaap(CIK, USER_AGENT)
print(f"  {len(gaap)} GAAP concepts available")

# Income statement
revenue      = (flow_qtr(gaap, "RevenueFromContractWithCustomerExcludingAssessedTax", "revenue")
                .combine_first(flow_qtr(gaap, "SalesRevenueNet", "revenue")))
gross_profit = flow_qtr(gaap, "GrossProfit",                             "gross_profit")
op_income    = flow_qtr(gaap, "OperatingIncomeLoss",                     "op_income")
net_income   = flow_qtr(gaap, "NetIncomeLoss",                           "net_income")
eps_dil      = flow_qtr(gaap, "EarningsPerShareDiluted",                 "eps_dil")
rd_exp       = flow_qtr(gaap, "ResearchAndDevelopmentExpense",           "rd_exp")
shares_dil   = flow_qtr(gaap, "WeightedAverageNumberOfDilutedSharesOutstanding", "shares_dil")

# Balance sheet
cash         = stock_qtr(gaap, "CashAndCashEquivalentsAtCarryingValue",  "cash")
equity       = stock_qtr(gaap, "StockholdersEquity",                     "equity")
total_assets = stock_qtr(gaap, "Assets",                                 "total_assets")

# Combine
fund = pd.concat([revenue, gross_profit, op_income, net_income,
                  eps_dil, rd_exp, shares_dil, cash, equity, total_assets], axis=1)
fund.index = pd.to_datetime(fund.index)
fund = fund[fund["revenue"].notna()].sort_index()

print(f"\nFundamental data: {len(fund)} quarters")
print(f"  Period : {fund.index[0].date()} → {fund.index[-1].date()}")
print(f"\nNon-null counts per metric:")
print(fund.notna().sum().to_string())

In [ ]:
# ── Price data ────────────────────────────────────────────────────────────────
print(f"Downloading {TICKER} daily price history from yfinance...")
amd_tk = yf.Ticker(TICKER)
px_raw = amd_tk.history(period="max", interval="1d")[["Close"]]
px_raw.index = pd.to_datetime(px_raw.index.tz_localize(None).date)
px_raw = px_raw.sort_index()

print(f"  Price data: {px_raw.index[0].date()} → {px_raw.index[-1].date()}")
print(f"  Trading days: {len(px_raw):,}")

def price_on(date):
    """Closing price on or before a given date (handles weekends / holidays)."""
    mask = px_raw.index <= date
    return float(px_raw.loc[px_raw.index[mask].max(), "Close"]) if mask.any() else np.nan

def qtr_vol(date, n_days=90):
    """Annualised volatility of daily returns in the n_days before a date."""
    lo   = date - pd.Timedelta(days=120)
    mask = (px_raw.index >= lo) & (px_raw.index < date)
    r    = px_raw["Close"][mask].pct_change().dropna()
    return float(r.std() * np.sqrt(252)) if len(r) >= 15 else np.nan

# Attach prices to fundamental table
fund["price"] = [price_on(d) for d in fund.index]
fund["vol"]   = [qtr_vol(d)  for d in fund.index]

print(f"\nPrice attached to {fund['price'].notna().sum()}/{len(fund)} quarters")

# Quick sanity-check plot
fig, ax = plt.subplots(figsize=(12, 3))
ax.semilogy(px_raw.index, px_raw["Close"], lw=0.8, color="steelblue")
ax.set_title("AMD Daily Close Price (log scale)", fontsize=13)
ax.set_ylabel("Price (USD)")
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature engineering ───────────────────────────────────────────────────────

# Profitability margins
fund["gross_margin"]  = fund["gross_profit"] / fund["revenue"]
fund["op_margin"]     = fund["op_income"]    / fund["revenue"]
fund["net_margin"]    = fund["net_income"]   / fund["revenue"]
fund["rd_intensity"]  = fund["rd_exp"]       / fund["revenue"]

# Balance-sheet ratios
fund["cash_to_rev"]   = fund["cash"]         / fund["revenue"]
fund["asset_turns"]   = (fund["revenue"] * 4) / fund["total_assets"]   # annualised
fund["equity_ratio"]  = fund["equity"]       / fund["total_assets"]

# Growth rates (QoQ and YoY)
fund["rev_qoq"]       = fund["revenue"].pct_change(1)
fund["rev_yoy"]       = fund["revenue"].pct_change(4)
fund["eps_yoy"]       = fund["eps_dil"].pct_change(4)
fund["opinc_yoy"]     = fund["op_income"].pct_change(4)
fund["gm_qoq"]        = fund["gross_margin"].diff(1)
fund["gm_yoy"]        = fund["gross_margin"].diff(4)

# Valuation ratios
fund["mktcap"]        = fund["price"] * fund["shares_dil"]
fund["ps_ratio"]      = fund["mktcap"] / (fund["revenue"] * 4)
fund["pb_ratio"]      = fund["mktcap"] / fund["equity"].abs()
# P/E only when earnings are positive (avoids sign inversion)
fund["pe_ratio"]      = np.where(
    fund["eps_dil"] > 0,
    fund["price"] / (fund["eps_dil"] * 4),
    np.nan
)
# Binary flag: profitable this quarter
fund["is_profitable"] = (fund["eps_dil"] > 0).astype(float)

# Price momentum
fund["mom_1q"]        = fund["price"].pct_change(1)
fund["mom_2q"]        = fund["price"].pct_change(2)
fund["mom_4q"]        = fund["price"].pct_change(4)

# ── Target: next-quarter price return ─────────────────────────────────────────
# return(Q+1) = price(Q+1) / price(Q) - 1
# Shift price return backward by 1 row so each row's target = NEXT quarter's return
fund["next_q_return"] = fund["price"].pct_change(1).shift(-1)

# ── Build clean model dataset ─────────────────────────────────────────────────
# P/E and some others are optional — use them only if they don't shrink the dataset too much
BASE_FEATURES = [
    "gross_margin", "op_margin", "net_margin", "rd_intensity",
    "rev_qoq", "rev_yoy", "eps_yoy", "opinc_yoy", "gm_qoq", "gm_yoy",
    "cash_to_rev", "asset_turns", "equity_ratio",
    "ps_ratio", "pb_ratio", "is_profitable",
    "mom_1q", "mom_2q", "mom_4q", "vol",
]

TARGET = "next_q_return"
data   = fund[BASE_FEATURES + [TARGET, "price", "revenue"]].dropna().copy()

# Winsorise at 2nd / 98th percentile to reduce outlier influence
for col in BASE_FEATURES:
    lo, hi = data[col].quantile([0.02, 0.98])
    data[col] = data[col].clip(lower=lo, upper=hi)

FEATURE_COLS = BASE_FEATURES
print(f"Model dataset  : {len(data)} quarters")
print(f"Period         : {data.index[0].date()} → {data.index[-1].date()}")
print(f"Features       : {len(FEATURE_COLS)}")
print(f"Target (mean)  : {data[TARGET].mean():.3f}  (std: {data[TARGET].std():.3f})")
print(f"Target positive: {(data[TARGET] > 0).mean():.1%} of quarters")

In [ ]:
# ── Exploratory Data Analysis ─────────────────────────────────────────────────

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("AMD Quarterly Fundamentals (model period)", fontsize=14, y=1.01)

metrics = [
    ("revenue",       "Revenue ($B)",      1e9),
    ("gross_margin",  "Gross Margin",      1),
    ("op_margin",     "Operating Margin",  1),
    ("rev_yoy",       "Revenue YoY Growth",1),
    ("ps_ratio",      "Price / Sales",     1),
    ("next_q_return", "Next-Qtr Return",   1),
]

for ax, (col, title, scale) in zip(axes.flat, metrics):
    vals = fund[col].dropna() / scale
    ax.plot(vals.index, vals, marker="o", ms=3, lw=1.2)
    ax.axhline(0, color="gray", lw=0.8, ls="--")
    ax.set_title(title, fontsize=11)
    ax.xaxis.set_major_formatter(mdates := __import__("matplotlib.dates", fromlist=["DateFormatter"]).DateFormatter("%Y"))
    ax.xaxis.set_major_locator(__import__("matplotlib.dates", fromlist=["YearLocator"]).YearLocator(2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")

plt.tight_layout()
plt.show()

# Distribution of next-quarter returns
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(data[TARGET], bins=20, color="steelblue", edgecolor="white", alpha=0.85)
ax1.axvline(0, color="red", lw=1.5, ls="--", label="0%")
ax1.set_title("Distribution of Next-Quarter Returns", fontsize=12)
ax1.set_xlabel("Return"); ax1.legend()

# Correlation heatmap
corr = data[FEATURE_COLS + [TARGET]].corr()[[TARGET]].drop(TARGET).sort_values(TARGET)
colors = ["#d73027" if v < 0 else "#1a9850" for v in corr[TARGET]]
ax2.barh(corr.index, corr[TARGET], color=colors, alpha=0.8)
ax2.axvline(0, color="black", lw=0.8)
ax2.set_title(f"Feature Correlation with {TARGET}", fontsize=12)
ax2.set_xlabel("Pearson r")
plt.tight_layout()
plt.show()

In [ ]:
# ── Walk-forward model training ───────────────────────────────────────────────
#
# At each step i (i >= MIN_TRAIN):
#   - Train on quarters  0 … i-1
#   - Predict quarter    i
#   - Record prediction and actual
# No future data ever touches the training window → zero look-ahead.

MODELS = {
    "Naive Mean": None,   # baseline: predict historical mean
    "Ridge": Pipeline([
        ("sc", StandardScaler()),
        ("m",  RidgeCV(alphas=[0.1, 1, 10, 100, 1000])),
    ]),
    "Random Forest": Pipeline([
        ("sc", StandardScaler()),
        ("m",  RandomForestRegressor(
            n_estimators=300, max_depth=4, min_samples_leaf=4,
            max_features=0.6, random_state=RANDOM_STATE)),
    ]),
    "XGBoost": Pipeline([
        ("sc", StandardScaler()),
        ("m",  xgb.XGBRegressor(
            n_estimators=150, max_depth=3, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.7,
            min_child_weight=5, reg_lambda=2.0,
            random_state=RANDOM_STATE, verbosity=0)),
    ]),
}

X      = data[FEATURE_COLS].values
y      = data[TARGET].values
dates  = data.index
records = {name: [] for name in MODELS}

for i in range(MIN_TRAIN, len(data) - 1):
    Xtr, ytr = X[:i],    y[:i]
    Xte      = X[i:i+1]
    yte      = y[i]
    date_i   = dates[i]

    # Naive baseline
    records["Naive Mean"].append({"date": date_i, "y_true": yte, "y_pred": float(ytr.mean())})

    for name, pipe in MODELS.items():
        if pipe is None:
            continue
        p = clone(pipe)
        p.fit(Xtr, ytr)
        records[name].append({"date": date_i, "y_true": yte, "y_pred": float(p.predict(Xte)[0])})

# ── Metrics ────────────────────────────────────────────────────────────────────
results_df = {}
print(f"\n{'Model':<16} {'MAE':>7} {'RMSE':>7} {'DirAcc':>8} {'IC':>7}")
print("─" * 52)
for name in MODELS:
    df_r = pd.DataFrame(records[name])
    mae  = mean_absolute_error(df_r["y_true"], df_r["y_pred"])
    rmse = mean_squared_error(df_r["y_true"],  df_r["y_pred"]) ** 0.5
    da   = (np.sign(df_r["y_true"]) == np.sign(df_r["y_pred"])).mean()
    ic   = df_r["y_true"].corr(df_r["y_pred"])
    results_df[name] = df_r
    print(f"{name:<16} {mae:7.4f} {rmse:7.4f} {da:8.2%} {ic:7.3f}")

print("\nDirAcc = direction accuracy (% of quarters where sign was correct)")
print("IC     = information coefficient (Pearson corr between predicted and actual)")

In [ ]:
# ── Visualise results ─────────────────────────────────────────────────────────

best_model = "XGBoost"   # change if another model performed better above
df_best    = results_df[best_model]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ── Plot 1: Actual vs Predicted ──────────────────────────────────────────────
ax = axes[0]
ax.scatter(df_best["y_pred"], df_best["y_true"], alpha=0.6, s=40,
           color="steelblue", edgecolors="white", lw=0.5)
lim = max(abs(df_best[["y_true","y_pred"]].values).max() * 1.1, 0.3)
ax.plot([-lim, lim], [-lim, lim], "r--", lw=1, label="Perfect prediction")
ax.axhline(0, color="gray", lw=0.6, ls=":")
ax.axvline(0, color="gray", lw=0.6, ls=":")
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.set_xlabel("Predicted Return"); ax.set_ylabel("Actual Return")
ax.set_title(f"{best_model}: Predicted vs Actual (walk-forward)", fontsize=12)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend()

# ── Plot 2: Cumulative returns — strategy vs buy-and-hold ───────────────────
ax2 = axes[1]
# Strategy: go long when model predicts positive return, else stay in cash
df_best = df_best.sort_values("date").copy()
df_best["strategy_ret"] = df_best["y_true"] * (df_best["y_pred"] > 0).astype(float)
df_best["bnh_ret"]      = df_best["y_true"]

df_best["cum_strategy"] = (1 + df_best["strategy_ret"]).cumprod()
df_best["cum_bnh"]      = (1 + df_best["bnh_ret"]).cumprod()

ax2.plot(df_best["date"], df_best["cum_strategy"], lw=2, label=f"{best_model} (long-only signal)")
ax2.plot(df_best["date"], df_best["cum_bnh"],      lw=2, ls="--", label="Buy & Hold")
ax2.axhline(1, color="gray", lw=0.8, ls=":")
ax2.set_title("Cumulative Returns: Signal vs Buy-and-Hold", fontsize=12)
ax2.set_ylabel("Cumulative Return (start = 1)")
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax2.legend(); ax2.xaxis.set_major_formatter(__import__("matplotlib.dates", fromlist=["DateFormatter"]).DateFormatter("%Y"))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha="right")

plt.tight_layout()
plt.show()

# Sharpe ratio of strategy vs buy & hold
def sharpe(ret_series, periods_per_year=4):
    mu  = ret_series.mean() * periods_per_year
    sig = ret_series.std()  * np.sqrt(periods_per_year)
    return mu / sig if sig > 0 else np.nan

print("\nAnnualised Sharpe ratios (approximate, no risk-free rate):")
print(f"  {best_model} signal : {sharpe(df_best['strategy_ret']):.2f}")
print(f"  Buy & Hold        : {sharpe(df_best['bnh_ret']):.2f}")

In [ ]:
# ── Feature importance ────────────────────────────────────────────────────────

# Re-fit on the full dataset to get importance over all available data
X_all = data[FEATURE_COLS].values
y_all = data[TARGET].values

rf_full = clone(MODELS["Random Forest"])
rf_full.fit(X_all, y_all)

xgb_full = clone(MODELS["XGBoost"])
xgb_full.fit(X_all, y_all)

rf_imp  = rf_full.named_steps["m"].feature_importances_
xgb_imp = xgb_full.named_steps["m"].feature_importances_

imp_df = pd.DataFrame({
    "feature":       FEATURE_COLS,
    "RandomForest":  rf_imp,
    "XGBoost":       xgb_imp,
}).set_index("feature")
imp_df["mean"] = imp_df.mean(axis=1)
imp_df = imp_df.sort_values("mean", ascending=True)

fig, ax = plt.subplots(figsize=(9, 8))
y_pos = range(len(imp_df))
ax.barh(y_pos, imp_df["RandomForest"], alpha=0.65, label="Random Forest", color="#4878d0")
ax.barh(y_pos, imp_df["XGBoost"],     alpha=0.65, label="XGBoost",       color="#ee854a")
ax.set_yticks(list(y_pos))
ax.set_yticklabels(imp_df.index, fontsize=10)
ax.set_xlabel("Feature Importance (normalised)")
ax.set_title("Feature Importance — RF vs XGBoost (full-sample fit)", fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

print("\nTop 5 features by average importance:")
print(imp_df["mean"].sort_values(ascending=False).head(5).to_string())

In [ ]:
# ── Next-quarter forecast (train on all available data) ───────────────────────

X_train = data[FEATURE_COLS].values
y_train = data[TARGET].values

forecasters = {}
for name, pipe in MODELS.items():
    if pipe is None:
        continue
    p = clone(pipe)
    p.fit(X_train, y_train)
    forecasters[name] = p

# Latest quarter features (last row of fund with all features)
latest_row = data[FEATURE_COLS].iloc[-1]
latest_date = data.index[-1]

print(f"Forecasting from: {latest_date.date()} (most recent quarter with full data)")
print(f"Forecast for    : Q ending ~{(latest_date + pd.DateOffset(months=3)).date()}")
print()
print(f"{'Model':<16} {'Predicted Return':>18}")
print("─" * 38)
preds = {}
for name, p in forecasters.items():
    pred = p.predict(latest_row.values.reshape(1, -1))[0]
    preds[name] = pred
    direction = "▲ UP" if pred > 0 else "▼ DOWN"
    print(f"{name:<16}   {pred:>+.2%}  {direction}")

ensemble_pred = np.mean(list(preds.values()))
print(f"\n{'Ensemble (mean)':<16}   {ensemble_pred:>+.2%}  {'▲ UP' if ensemble_pred > 0 else '▼ DOWN'}")

print()
print("─" * 60)
print("NOTE: This model is trained on ~30 quarters of data.")
print("Treat predictions as one input among many — not a trading signal.")